# 01 — Explore Ground Truth

**Owner:** Part 1 (Setup & Data Foundation)

Purpose: this notebook is the Day-1 "trace 3 items by hand" exercise from the
project guide, done properly and left as a reusable artifact for the rest of
the team. It:

1. Loads `Unilog-Sample_200_Items-Input-vs-Output.xlsx` (Input + Delivery
   Format sheets) via the shared `test_harness` loader.
2. Traces a few full items across, left to right, input → output.
3. Scans for placeholder values (`-- Unbranded --` etc.) using
   `placeholder_utils`.
4. Surfaces the ground truth's own known gaps (blank UNSPSC / country-of-
   origin, manufacturer/brand mismatches) rather than hiding them.
5. Does a first look at the reference index, UOM table, and
   Decimal_Fraction table's 4-block layout, so no one downstream is
   surprised by the messy-sheet quirks mentioned in the project guide.

This notebook does not build the full lookup tables (that's
`lookups.py` / a later notebook) — it's exploration only.

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the shared src/ package importable from notebooks/
sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd

from unihack.part1_foundation.test_harness import (
    load_ground_truth,
    TestHarness,
    DEFAULT_INPUT_SHEET,
    DEFAULT_DELIVERY_SHEET,
)
from unihack.part1_foundation.placeholder_utils import (
    scan_column,
    summarize_placeholders,
    strip_placeholders,
    KNOWN_PLACEHOLDERS,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

In [ ]:
# Point this at the real pack file once it's dropped into data/raw/.
# Falls back to building a tiny synthetic workbook so this notebook still
# runs end-to-end before the real data is available.
DATA_DIR = Path("..") / "data" / "raw"
GROUND_TRUTH_PATH = DATA_DIR / "Unilog-Sample_200_Items-Input-vs-Output.xlsx"

USING_SYNTHETIC_DATA = not GROUND_TRUTH_PATH.exists()
if USING_SYNTHETIC_DATA:
    print(
        f"[!] {GROUND_TRUTH_PATH} not found -- building a small synthetic "
        "workbook so this notebook still runs. Re-run this notebook once "
        "the real pack file is in data/raw/."
    )
GROUND_TRUTH_PATH, USING_SYNTHETIC_DATA

In [ ]:
def _build_synthetic_ground_truth(path: Path) -> None:
    """Small stand-in for the real 200-item file, used only if it's missing."""
    input_df = pd.DataFrame(
        {
            "SKU": ["SKU001", "SKU002", "SKU003"],
            "Part_Desc": [
                "PDSH4816AF Dishwasher SS - Display Only",
                "3/8 CPLG BRS 150#",
                "1/2 GATE VLV BRS 200 WOG",
            ],
            "Unilog_Brand": ["FRIGIDAIRE", "-- Unbranded --", "ACME"],
        }
    )
    output_df = pd.DataFrame(
        {
            "SKU": ["SKU001", "SKU002", "SKU003"],
            "Classpath": [
                "Appliances & Consumer Electronics > Kitchen Appliances > Built-In Dishwashers",
                "Plumbing > Fittings > Couplings",
                "Plumbing > Valves > Gate Valves",
            ],
            "Brand": ["FRIGIDAIRE®", None, "ACME Corp"],
            "Product Title": [
                "FRIGIDAIRE® Professional Series PDSH4816AF Dishwasher With CleanBoost(TM)",
                "3/8 in Brass Coupling, 150 PSI",
                "1/2 in Brass Gate Valve, 200 WOG",
            ],
            "Invoice Desc": [
                "DISHWASHER LEG 5 SST 120V 15A 50-1/4IN",
                "CPLG BRS 3/8IN 150#",
                "GATE VLV BRS 1/2IN 200WOG",
            ],
            "UNSPSC": ["52141500", None, "40141700"],
            "Country of Origin": ["USA", "USA", None],
        }
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        input_df.to_excel(writer, sheet_name=DEFAULT_INPUT_SHEET, index=False)
        output_df.to_excel(writer, sheet_name=DEFAULT_DELIVERY_SHEET, index=False)


if USING_SYNTHETIC_DATA:
    synthetic_path = Path("_synthetic_200_items.xlsx")
    _build_synthetic_ground_truth(synthetic_path)
    GROUND_TRUTH_PATH = synthetic_path

## 1. Load ground truth via the shared test harness

In [ ]:
gt_records = load_ground_truth(GROUND_TRUTH_PATH)
harness = TestHarness(gt_records)

print(f"Loaded {len(harness)} ground-truth records.")
print(f"Input fields per record (sample): {list(gt_records[0].input_fields.keys())}")
print(f"Expected/output fields per record (sample): {list(gt_records[0].expected_fields.keys())}")

## 2. Trace items by hand, left to right

The project guide is explicit that tracing a few full items across, input →
output, teaches more than reading any spec doc. Do that here for the first
3 records.

In [ ]:
N_TO_TRACE = 3

for record in gt_records[:N_TO_TRACE]:
    print("=" * 100)
    print(f"KEY: {record.key}")
    print("-" * 100)
    print("INPUT FIELDS:")
    for k, v in record.input_fields.items():
        print(f"  {k:20s} = {v!r}")
    print("-" * 100)
    print("EXPECTED (Delivery Format) FIELDS:")
    for k, v in record.expected_fields.items():
        print(f"  {k:20s} = {v!r}")
    if record.known_gaps:
        print("-" * 100)
        print(f"KNOWN SOURCE GAPS (blank in ground truth itself): {sorted(record.known_gaps)}")
    print()

### Observation

The same underlying product (row 1 in the real 200-item file) gets rewritten
into ~5 different output strings — invoice, mobile, title, long description,
and marketing copy — each with its own length limit and casing rule. That's
the core of the description-building stage (Part 3): a constrained
formatting/construction problem, not a "write a nice description" problem.

Note anything surprising here before moving on — e.g. which fields are
present on Input but not obviously reused verbatim in the output, and which
output fields have no obvious Input-side source (implying they come from
attribute extraction / enrichment, not straight copy-through).

## 3. Scan for placeholder values in brand/manufacturer fields

In [ ]:
brand_like_cols_input = [
    c for c in pd.DataFrame([r.input_fields for r in gt_records]).columns
    if pd.api.types.is_string_dtype(pd.DataFrame([r.input_fields for r in gt_records])[c])
    and (("brand" in c.lower()) or ("manuf" in c.lower()))
]
input_df_full = pd.DataFrame([r.input_fields for r in gt_records])

print(f"Detected brand/manufacturer-like input columns: {brand_like_cols_input}")
summarize_placeholders(input_df_full, columns=brand_like_cols_input)

In [ ]:
# Per-column detail: which unlisted placeholder-shaped values (if any) turned
# up that AREN'T in KNOWN_PLACEHOLDERS -- these need a human look before
# being folded into the known list.
for col in brand_like_cols_input:
    report = scan_column(input_df_full[col], col)
    if report.unlisted_examples:
        print(f"[{col}] unlisted placeholder-shaped values found: {report.unlisted_examples}")
    else:
        print(f"[{col}] no unlisted placeholder-shaped values -- known list looks sufficient.")

## 4. Surface known gaps in the ground truth (don't hide them)

Per the project rules: blank UNSPSC / country-of-origin cells, and any
manufacturer/brand mismatch row, are gaps in the ground truth itself, not
pipeline errors. Detecting and flagging these is a strength to show off, not
something to quietly filter out.

In [ ]:
gaps_report = harness.known_issues_report()
gaps_report

In [ ]:
if gaps_report.empty:
    print("No known gaps detected in this run (expected if using the synthetic fallback with no gaps).")
else:
    gap_field_counts = (
        gaps_report["known_gaps"]
        .explode()
        .value_counts()
        .rename("rows_with_this_gap")
    )
    print("Gap frequency by field:")
    print(gap_field_counts)

## 5. First look at the reference index and messy-sheet quirks

Quick sanity checks for the other pack files, so their layout quirks are
documented here once rather than rediscovered by every downstream stage.
Adjust paths once the real files are in `data/raw/`.

In [ ]:
reference_index_path = DATA_DIR / "Reference_Documents_Summary.xlsx"
if reference_index_path.exists():
    ref_index = pd.read_excel(reference_index_path)
    display(ref_index)
else:
    print(f"[!] {reference_index_path} not found yet -- open it manually per Step 1.1 of the guide "
          "once it's dropped into data/raw/, this is a 5-minute non-negotiable read.")

In [ ]:
decimal_fraction_path = DATA_DIR / "Decimal_Fraction.xlsx"
if decimal_fraction_path.exists():
    raw = pd.read_excel(decimal_fraction_path, header=None)
    print(f"Raw shape: {raw.shape}")
    print("First 5 rows, all columns (confirm the 4 side-by-side Fraction|Decimal blocks):")
    display(raw.head())
else:
    print(
        f"[!] {decimal_fraction_path} not found yet. "
        "REMINDER (per project guide + Part 1 pitfalls list): this file is "
        "laid out as 4 side-by-side Fraction|Decimal column blocks, NOT one "
        "column -- read with header=None first and inspect manually before "
        "assuming a simple 2-column shape."
    )

In [ ]:
uom_path = DATA_DIR / "Unilog_Master_UOM_Standards_Abbreviations_and_Terms.xlsx"
if uom_path.exists():
    uom_xls = pd.ExcelFile(uom_path)
    print(f"Sheets found: {uom_xls.sheet_names}")
    for sheet in uom_xls.sheet_names:
        df = pd.read_excel(uom_xls, sheet_name=sheet, nrows=5)
        print(f"\n--- {sheet} (first 5 rows) ---")
        display(df)
else:
    print(
        f"[!] {uom_path} not found yet. "
        "REMINDER: Sheet2 house-style rules are reportedly parked in stray "
        "columns -- don't assume row 1 is a clean header on this file."
    )

## 6. Handoff notes (fill in once run against the real pack file)

- [ ] Confirmed join key between Input and Delivery Format sheets (currently: SKU)
- [ ] Confirmed no unlisted placeholder variants beyond `KNOWN_PLACEHOLDERS`
      (or listed the ones found above for review)
- [ ] Confirmed which fields are legitimately blank in ground truth
      (UNSPSC, Country of Origin, ...) vs. which are pipeline bugs
- [ ] Confirmed Decimal_Fraction.xlsx's 4-block layout and noted exact
      column names/offsets for the parser
- [ ] Confirmed UOM sheet structure (Sheet1 vs Sheet2, any stray-column notes)
- [ ] Anything else surprising, documented in `docs/scope_notes.md`